[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_19_minibatch_sgd_scan.ipynb)

# 🔴 Hard: Mini-batch SGD with nested lax.scan

*Training*
Train linear regression with **mini-batch SGD**, where the epoch loop and the
batch loop are both `lax.scan`, and the PRNG key is part of the carry.

### Signature
```python
def sgd_epochs(X, y, key, lr=0.1, batch_size=10, epochs=20):
    ...   # -> (w, b, losses, key)
```

`X` is `(N, D)`, `y` is `(N,)`. Start at `w = 0`, `b = 0`. Returns:

- `w` `(D,)`, `b` scalar — the trained parameters
- `losses` `(epochs,)` — the mean of that epoch's batch losses
- `key` — the key **after** training, so the caller can resume

Gradients are the same hand-derived MSE ones as problem 40 and `b_18`, but
computed on each mini-batch:

$$\nabla_w = \frac{2}{B}X_b^\top(\hat{y}_b-y_b), \qquad
  \nabla_b = \frac{2}{B}\sum(\hat{y}_b-y_b)$$

### The two scans

**Outer — epochs.** No per-step input, so `xs=None` and `length=epochs`.
Its carry is `((w, b), key)`.

**Inner — batches.** Here there *is* a per-step input: the batches themselves.
Reshape the shuffled data to `(n_batches, batch_size, D)` and hand it to scan
as `xs`; scan walks the leading axis, so the body sees one batch at a time.
Its carry is just `(w, b)`.

```python
(w, b), batch_losses = jax.lax.scan(batch_body, (w, b), (Xs, ys))
```

`n_batches = N // batch_size`, and the remainder is **dropped** — every shape
inside a scan has to be static, and a short final batch is not.

### Why returning the key is part of the exercise
Splitting inside the loop is the whole point:

```python
key, sub = jax.random.split(key)   # shuffle with sub, carry key onward
perm = jax.random.permutation(sub, N)
```

Reuse the incoming `key` every epoch instead and **the model still converges** —
you get the same permutation each time, which on a small problem costs you
almost nothing measurable. That is exactly what makes it dangerous: a silent
bug that only shows up as a mysteriously worse model months later. Returning
the final key makes it visible, and it is what a real training loop does
anyway, because you need somewhere to resume from.

### What is hard here
1. **The carry structure must match exactly** — same pytree in and out. Nest
   it as `((w, b), key)` and keep it nested; returning `(w, b, key)` from the
   body will not match the init.
2. **The key must advance.** Split, use the child, carry the parent.
3. **`batch_size` and `epochs` are static.** They set the shapes of `xs` and
   of `losses`, so under `jit` they are `static_argnames`.
4. **Permute `X` and `y` with the SAME indices.** Drawing two permutations, or
   shuffling only `X`, destroys the pairing and the fit never converges.

### Where you have met the pieces
`b_06` and `b_18` are the carry; `b_05` is key splitting; the difference here
is that they have to work at the same time, one nested inside the other.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def sgd_epochs(X, y, key, lr=0.1, batch_size=10, epochs=20):
    """Mini-batch SGD with both loops as lax.scan.

    Returns (w, b, losses, key):
      w       (D,)        trained weights, started from zeros
      b       scalar      trained bias, started from 0
      losses  (epochs,)   mean batch loss for each epoch
      key                 the key AFTER training, for resuming
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

X = jax.random.normal(jax.random.key(0), (100, 3))
w_true = jnp.array([2.0, -1.0, 0.5])
y = X @ w_true + 0.3

key = jax.random.key(42)
w, b, losses, key2 = sgd_epochs(X, y, key, lr=0.1, batch_size=10, epochs=50)

print(f"w = {jnp.round(w, 4)}   truth {w_true}")
print(f"b = {float(b):.4f}      truth 0.3")
print(f"losses {losses.shape}: {float(losses[0]):.4f} -> {float(losses[-1]):.3e}")

# The key came back advanced, so training can pick up where it left off.
same = jnp.array_equal(jax.random.key_data(key), jax.random.key_data(key2))
print(f"\nkey advanced? {not same}")
w2, b2, l2, _ = sgd_epochs(X, y, key2, lr=0.1, batch_size=10, epochs=50)
print(f"resumed from the returned key, first epoch loss {float(l2[0]):.3e}")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("minibatch_sgd_scan")

# hint("minibatch_sgd_scan")      # stuck? nudge without the answer
# solution("minibatch_sgd_scan")  # spoiler: the reference implementation
# status()                        # your dashboard across all problems